# Plotting Final — Cross-Series Historical State Comparison

Purpose: apply the **same observation month and same plotting method to every cleaned economic series**. The common reference is **May 1995**. Each series is plotted over its full available history; May 1995 is highlighted where that series has an observation in that calendar month. All series use the same method: full-series z-score plus raw values with trailing rolling means of 2 and 4 observations.

Important: z-scores use each series' full-history mean and sample standard deviation, so this is retrospective/descriptive and includes future observations relative to May 1995. It is not a no-look-ahead test. Different series can have different units and frequencies; standardization helps compare relative deviations, while rolling means are measured in each series' own observations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ---- Shared experiment controls: identical for every series ----
TARGET_YEAR = 1995
TARGET_MONTH = 5
ROLLING_WINDOWS = (2, 4)

# Locate data regardless of whether Jupyter starts in repository root or notebook folder.
candidates = [Path('data/cleaned'), Path('../data/cleaned'), Path('../../data/cleaned')]
cleaned_dir = next((p.resolve() for p in candidates if p.exists()), None)
if cleaned_dir is None:
    raise FileNotFoundError('Cannot find data/cleaned. Start Jupyter from repository root or notebook folder.')

csv_paths = sorted(cleaned_dir.glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(f'No CSV files found in {cleaned_dir}')

month_start = pd.Timestamp(TARGET_YEAR, TARGET_MONTH, 1)
month_end = month_start + pd.offsets.MonthEnd(1)
print(f'Common reference month: {month_start:%B %Y}')
print(f'Cleaned CSV files discovered: {len(csv_paths)}')
print(f'Rolling windows for every series: {ROLLING_WINDOWS}')

# Load every series using the same schema and cleaning rules.
series_by_id = {}
load_issues = []
for csv_path in csv_paths:
    series_id = csv_path.stem
    try:
        raw = pd.read_csv(csv_path)
        if not {'date', 'value'}.issubset(raw.columns):
            load_issues.append((series_id, f"Missing date/value columns: {list(raw.columns)}"))
            continue
        raw['date'] = pd.to_datetime(raw['date'], errors='coerce')
        raw['value'] = pd.to_numeric(raw['value'], errors='coerce')
        s = (raw.dropna(subset=['date', 'value'])
                 .sort_values('date')
                 .drop_duplicates('date', keep='last')
                 .set_index('date')['value'])
        if s.empty:
            load_issues.append((series_id, 'No valid date/value observations'))
            continue
        series_by_id[series_id] = s
    except Exception as exc:
        load_issues.append((series_id, str(exc)))

if not series_by_id:
    raise ValueError('No valid economic series could be loaded.')

print(f'Valid series loaded: {len(series_by_id)}')
if load_issues:
    print(f'Series skipped due to data/schema issues: {len(load_issues)}')
    display(pd.DataFrame(load_issues, columns=['series_id', 'issue']))

# Build a consistent summary for every series at the shared reference month.
summary_rows = []
for series_id, s in series_by_id.items():
    target_mask = (s.index >= month_start) & (s.index <= month_end)
    ref_mean = s.mean()
    ref_std = s.std(ddof=1)
    target_values = s.loc[target_mask]
    target_z = ((target_values - ref_mean) / ref_std) if pd.notna(ref_std) and ref_std != 0 else pd.Series(dtype=float)
    summary_rows.append({
        'series_id': series_id,
        'first_date': s.index.min(),
        'last_date': s.index.max(),
        'observations': len(s),
        'observations_in_may_1995': int(target_mask.sum()),
        'may_1995_value': target_values.iloc[0] if len(target_values) else np.nan,
        'may_1995_full_series_z_score': target_z.iloc[0] if len(target_z) else np.nan,
        'full_series_mean': ref_mean,
        'full_series_sample_std': ref_std
    })

summary = pd.DataFrame(summary_rows).sort_values('series_id').reset_index(drop=True)
display(summary)


## One consistent visualization method, applied to every economic series

For each series, the next cell creates two full-history panels: (1) full-series z-score and (2) raw observed values with 2- and 4-observation trailing rolling means. The orange band marks May 1995. If a series has no observation dated within May 1995 (for example, because of its reporting frequency or coverage), its chart still uses the same reference month but reports that no observation falls inside the highlighted window.

In [ ]:
# Apply the same plotting procedure to every cleaned economic series.
for series_id, s in sorted(series_by_id.items()):
    target_mask = (s.index >= month_start) & (s.index <= month_end)
    ref_mean = s.mean()
    ref_std = s.std(ddof=1)
    rolling_2 = s.rolling(window=ROLLING_WINDOWS[0], min_periods=ROLLING_WINDOWS[0]).mean()
    rolling_4 = s.rolling(window=ROLLING_WINDOWS[1], min_periods=ROLLING_WINDOWS[1]).mean()

    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, constrained_layout=True)
    for ax in axes:
        ax.axvspan(month_start, month_end, color='orange', alpha=0.2, label=f'Reference: {month_start:%B %Y}')

    if pd.notna(ref_std) and ref_std != 0:
        z = (s - ref_mean) / ref_std
        axes[0].plot(z.index, z.values, linewidth=1.0, label='Full-series z-score')
        axes[0].axhline(0, color='black', linewidth=1, label='Full-series mean')
        axes[0].axhline(1, linestyle='--', alpha=0.5, label='+/- 1 std')
        axes[0].axhline(-1, linestyle='--', alpha=0.5)
        axes[0].axhline(2, linestyle=':', alpha=0.5, label='+/- 2 std')
        axes[0].axhline(-2, linestyle=':', alpha=0.5)
        if target_mask.any():
            target_z = z.loc[target_mask]
            axes[0].scatter(target_z.index, target_z.values, color='darkorange', zorder=5, label='May 1995 observation')
        axes[0].set_ylabel('Z-score')
        axes[0].legend(fontsize=7, ncol=3)
    else:
        axes[0].text(0.5, 0.5, 'Z-score unavailable (constant or insufficient data)',
                     transform=axes[0].transAxes, ha='center', va='center')
        axes[0].set_ylabel('Z-score')

    axes[0].set_title(f'{series_id} — standardized full history | common reference: May 1995')
    axes[0].grid(True, alpha=0.2)

    axes[1].plot(s.index, s.values, linewidth=1.0, label='Observed value')
    axes[1].plot(rolling_2.index, rolling_2.values, linewidth=1.2, label='Rolling mean (2 observations)')
    axes[1].plot(rolling_4.index, rolling_4.values, linewidth=1.5, label='Rolling mean (4 observations)')
    if target_mask.any():
        axes[1].scatter(s.loc[target_mask].index, s.loc[target_mask].values,
                        color='darkorange', zorder=5, label='May 1995 observation')
    axes[1].set_title('Raw series with the same rolling-mean method')
    axes[1].set_ylabel('Original units')
    axes[1].set_xlabel('Observation date')
    axes[1].legend(fontsize=7, ncol=2)
    axes[1].grid(True, alpha=0.2)
    plt.show()

    if target_mask.any():
        print(f'{series_id}: observation(s) in May 1995')
        display(s.loc[target_mask].rename('value').to_frame())
    else:
        print(f'{series_id}: NO observation dated within May 1995; common reference window shown for consistency.')


## Interpretation checklist

- Compare the standardized position of each series around May 1995, rather than comparing raw values with incompatible units.
- Check which series actually contain an observation during May 1995; the summary table makes missing coverage/frequency explicit.
- The rolling windows mean 2 and 4 **observations**, not necessarily 2 and 4 months. Their calendar span depends on each series' frequency.
- Full-series z-scores use the entire history, including observations after May 1995. Treat this as retrospective exploration, not a point-in-time trading signal.
- A visual deviation is not, by itself, evidence of mean reversion, causality, or a tradable anomaly.